In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
project_root = Path.cwd().parent  # repository root (run notebooks from notebooks/)
obs_dir = project_root / "data" / "observations" / "q"
mapping_file = project_root / "data" / "observations" / "mapping.csv"

t_start = pd.Timestamp("2021-07-12 00:00")
t_end   = pd.Timestamp("2021-07-17 00:00")

mapping = pd.read_csv(mapping_file, sep=";")

obs_series = {}

for _, row in mapping.iterrows():

    station_name = row["name"]
    obs_id = str(row["id"])

    obs_files = list(obs_dir.glob(f"{obs_id}_*.csv"))
    if not obs_files:
        print(f"No obs file for {station_name}")
        continue

    # Read observations
    df = pd.read_csv(
        obs_files[0],
        sep=";",
        decimal=",",
        parse_dates=["Datum"],
        dayfirst=True,
        encoding="latin1"
    )

    # Filter time window
    df = df[
        (df["Datum"] >= t_start) &
        (df["Datum"] <= t_end)
    ]

    # Use time as index
    s = (
        df.set_index("Datum")["Wert"]
          .rename(station_name)
          .sort_index()
    )

    obs_series[station_name] = s

df_obs = pd.concat(obs_series.values(), axis=1)

df_obs.tail()

In [ ]:
# df_obs_Bad_Bodendorf = df_obs.loc[:, "Bad Bodendorf"] / 100

## Water Level Boundary Condition

Extract and process water level from Bad Bodendorf station. This creates a constant water level series for model boundary conditions.

In [ ]:
# df_obs_Bad_Bodendorf.plot()

In [ ]:
# 62.932

In [ ]:
# df_obs_Bad_Bodendorf.loc[:] = df_obs_Bad_Bodendorf.loc[:] + 62.932 # event try 1

In [ ]:
# df_obs_Bad_Bodendorf.loc[:] = df_obs_Bad_Bodendorf.loc[:] + 65.932 #-2.5 # event try 2 using the level from the website

In [ ]:
#df_obs_Bad_Bodendorf

In [ ]:
# df_obs_Bad_Bodendorf.loc[:] = 0.90 + 65.932 # this is the one from the website, this is from the model dep 66.67904

In [ ]:
#df_obs_Bad_Bodendorf

In [ ]:
#df_obs_Bad_Bodendorf.to_csv(project_root / "data" / "observations" / "water_level_ts_event_ref_web.csv", index=True)

In [ ]:
df_obs.loc[:, "Müsch"].plot()

In [ ]:
df_obs

In [ ]:
# df_obs.loc[:, "Kreuzberg"] =  1 # 0.33 in bnd 2 and 1
# df_obs.loc[:, "Denn"] = 1
# df_obs.loc[:, "Müsch"] = 3.5
# df_obs.loc[:, "Kirmutscheid"] = 1 # this only in the second try

# df_obs.loc[:, "Niederadenau"] = 0.33
# df_obs.loc[:, "Upstream1"] = 0.33
# df_obs.loc[:, "Upstream2"] = 0.33

## Discharge Source Points

Set constant discharge values at each source point for the simulation scenario (bnd9). These constant values are used as source/inflow forcing.

In [ ]:
# for bnd6
# df_obs.loc[:, "Kreuzberg"] =  5
# df_obs.loc[:, "Denn"] = 1.5
# df_obs.loc[:, "Müsch"] = 2
# df_obs.loc[:, "Kirmutscheid"] = 3 

# df_obs.loc[:, "Niederadenau"] = 0.17
# df_obs.loc[:, "Upstream1"] = 0.1
# df_obs.loc[:, "Upstream2"] = 0.1
# df_obs.loc[:, "Upstream3"] = 2.5

# for bnd7
# df_obs.loc[:, "Kreuzberg"] =  3.5
# df_obs.loc[:, "Denn"] = 1
# df_obs.loc[:, "Müsch"] = 2
# df_obs.loc[:, "Kirmutscheid"] = 3 

# df_obs.loc[:, "Niederadenau"] = 0.17
# df_obs.loc[:, "Upstream1"] = 0.1
# df_obs.loc[:, "Upstream2"] = 0.1
# df_obs.loc[:, "Upstream3"] = 2.5

# for bnd9 (disabled to keep time-varying discharge)
# df_obs.loc[:, "Kreuzberg"] =  3
# df_obs.loc[:, "Denn"] = 0.33
# df_obs.loc[:, "Müsch"] = 1
# df_obs.loc[:, "Kirmutscheid"] = 2.5 

# df_obs.loc[:, "Niederadenau"] = 0.17
# df_obs.loc[:, "Upstream1"] = 0.1
# df_obs.loc[:, "Upstream2"] = 0.1
# df_obs.loc[:, "Upstream3"] = 4

In [ ]:
df_obs.head()

In [ ]:
# df_obs.drop(columns=["Altenahr", "Bad Bodendorf", "Kirmutscheid"], inplace=True)
df_obs.drop(columns=["Altenahr", "Bad Bodendorf"], inplace=True)
#Altenahr and Bad Bodendorf are not used as source points, but can be used for validation. 

In [ ]:
name_to_index = {
    "Kreuzberg":    1,
    "Denn":         2,
    "Müsch":        4,
    "Kirmutscheid": 7,
    "New1":         10,
    "Niederadenau": 11,
    "New2":         12,
    "New3":         13,
}

In [ ]:
df_obs = df_obs.rename(columns=name_to_index)


In [ ]:
df_obs

In [ ]:
print(df_obs.columns.tolist())

we can see that the 3 tributaries (plus some correction ???) sum up to the altenahr 

In [ ]:
(obs_series["Altenahr"]).plot(label="Altenahr")
obs_series["Bad Bodendorf"].plot(label="Bad Bodendorf")
plt.legend()

In [ ]:
(obs_series["Müsch"] + obs_series["Denn"] + obs_series["Kreuzberg"] + obs_series["Kirmutscheid"]).plot(label="tribs")
obs_series["Altenahr"].plot(label="Altenahr")
plt.legend()


In [ ]:
import numpy as np
# --- Step 1: Estimate travel time lag via cross-correlation ---
gauged_names = ["Kreuzberg", "Denn", "Kirmutscheid", "Müsch"]
q_gauged_sum = sum(obs_series[n] for n in gauged_names).dropna()
q_altenahr   = obs_series["Altenahr"].dropna()

common_idx = q_gauged_sum.index.intersection(q_altenahr.index)
q_sum = q_gauged_sum.loc[common_idx]
q_ahr = q_altenahr.loc[common_idx]

# Cross-correlation to find lag between tributary sum and Altenahr
corr = np.correlate(
    q_ahr.values - q_ahr.mean(),
    q_sum.values - q_sum.mean(),
    mode="full"
)
lags     = np.arange(-(len(q_sum) - 1), len(q_sum))
best_lag = int(lags[np.argmax(corr)])   # timesteps (15-min intervals)
lag_hours = best_lag * 15 / 60

print(f"Best lag: {best_lag} timesteps = {lag_hours:.2f} hours")
print("(tributaries peak this many hours before Altenahr)")

# --- Step 2: Shift gauged sum forward by lag and compute residual ---
q_sum_shifted     = q_sum.shift(best_lag)
q_residual        = (q_ahr - q_sum_shifted).dropna()
q_residual_clipped = q_residual.clip(lower=0)

# Smooth with 1-hour rolling mean
q_residual_smooth = q_residual_clipped.rolling(4, center=True, min_periods=1).mean()

# --- Step 3: Distribute residual by watershed area ---
# Mapping: SFINCS source name -> (SFINCS index, watershed area [km²])
# Watershed IDs from QGIS; indices match the geojson IDs
ungauged = {
    "New1":         {"index": 10, "area": 60.771605},  # id: 10New1
    "Niederadenau": {"index": 11, "area": 61.338819},  # id: 11Niederadenau
    "New2":         {"index": 12, "area": 29.453133},  # id: 12New2
}

total_area = sum(v["area"] for v in ungauged.values())

# Shift residual back by 1.5 h (6 x 15-min steps) so that water injected at
# the source points arrives at Altenahr at the right time after routing
q_residual_injection = q_residual_smooth.shift(-6).bfill()

print(f"\nTotal ungauged area: {total_area:.2f} km²")
print(f"Max residual (total ungauged): {q_residual_injection.max():.2f} m³/s\n")

for name, meta in ungauged.items():
    frac = meta["area"] / total_area
    ts   = (q_residual_injection * frac).reindex(df_obs.index, method="nearest")
    df_obs[meta["index"]] = ts.values
    print(f"  {name:15s} (idx {meta['index']}): area={meta['area']:.2f} km²  "
          f"({frac*100:.1f}%)  peak={ts.max():.2f} m³/s")


# --- Step 4: Plot ---
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

for ax in axes:
    ax.grid(True)

ax = axes[0]
q_ahr.plot(ax=ax, label="Altenahr", color="k", linewidth=2)
q_sum.plot(ax=ax, label="Sum gauged (unshifted)", color="steelblue", linestyle="--", alpha=0.5)
q_sum_shifted.plot(ax=ax, label=f"Sum gauged (shifted +{lag_hours:.1f} h)", color="steelblue")
ax.set_ylabel("Discharge [m³/s]")
ax.set_title("Altenahr vs sum of gauged tributaries")
ax.legend()
ax.grid(True)

ax = axes[1]
q_residual.plot(ax=ax, label="Residual (raw)", color="orange", linestyle="--", alpha=0.5)
q_residual_smooth.plot(ax=ax, label="Residual (smoothed + clipped)", color="red", linewidth=2)
#for name, meta in ungauged.items():
#    col = meta["index"]
#    if col in df_obs.columns:
#        df_obs[col].plot(ax=ax, label=name, linestyle=":")
ax.axhline(0, color="k", linewidth=0.8, linestyle=":")
ax.set_ylabel("Discharge [m³/s]")
ax.set_title("Residual split between ungauged tributaries by watershed area")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

print(f"\ndf_obs columns now: {sorted(df_obs.columns.tolist())}")

In [ ]:
df_obs_timevarying = df_obs.copy()
df_obs_timevarying.to_csv(project_root / "data" / "observations" / "src_points_ts_Marg_additionalsrc.csv", index=True)

## Export and Verification

Save discharge time series to CSV and visualize combined flows from tributaries vs main Altenahr station.

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
df_obs.plot(ax=ax)

ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel("Discharge")

In [ ]:
(df_obs.sum(axis=1)).plot(label="Sum all tributaries")
obs_series["Altenahr"].plot(label="Altenahr")
plt.legend()

---
## Method 1 — Rainfall-weighted residual distribution

Improvement over the area-only method: distribute the Altenahr residual proportionally to **area × rainfall intensity** per tributary, and assign each tributary its own injection lag via the Kirpich formula.

$$Q_{\text{res},i}(t) = Q_\text{res}(t) \cdot \frac{A_i \cdot P_i(t)}{\sum_j A_j \cdot P_j(t)}$$

Injection lag (Kirpich): $t_c = 0.0663 \cdot \left(\frac{L}{\sqrt{S}}\right)^{0.77}$ [hours, with $L$ in km, $S$ dimensionless]

**Action required**: replace the placeholder $L$ and $S$ values in `kirpich_params` with values derived from DEM/OSM analysis for each subcatchment.

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Shared setup: load rainfall netCDF + extract mean rainfall per subcatchment
# (used by both Method 1 and Method 2)
# ─────────────────────────────────────────────────────────────────────────────

import geopandas as gpd
import rasterio.features
import rasterio.transform
import netCDF4 as nc
from pyproj import CRS, Transformer
from shapely.ops import transform as sh_transform
import warnings
warnings.filterwarnings("ignore")

# ── Catchment polygons (full delineated subcatchments, CRS EPSG:4326) ─────────
catchment_path = project_root / "data" / "catchment" / "ahr_catchment.geojson"
gdf_all = gpd.read_file(catchment_path)   # 27 features, matched by 'unitarea' column

ungauged = {
    "New1":         {"index": 10, "area_km2": 60.771605},   # COMID 23013430
    "Niederadenau": {"index": 11, "area_km2": 61.338819},   # COMID 23013317
    "New2":         {"index": 12, "area_km2": 29.453133},   # COMID 23013457
}
for name, m in ungauged.items():
    row = gdf_all[abs(gdf_all["unitarea"] - m["area_km2"]) < 0.01]
    m["geom_wgs84"] = row.geometry.iloc[0]

# ── Load precipitation netCDF ─────────────────────────────────────────────────
ds_nc    = nc.Dataset(project_root / "data" / "precip" / "RMI_RADFLOOD21_accum1h.nc")
x_nc     = ds_nc.variables["x"][:].data
y_nc     = ds_nc.variables["y"][:].data
time_h   = ds_nc.variables["time"][:].data    # hours since 2021-07-13 00:00
rain_raw = ds_nc.variables["precip"][:].data.astype(float)
rain_raw[rain_raw < 0] = 0.0

# RMI ACRR product: values in 0.01 mm (standard ODIM encoding) → divide by 100
rain_scale = 0.01    # → mm per 1-hour timestep
rain_mm    = rain_raw * rain_scale

# Parse CRS and build rasterio affine transform for the precipitation grid
precip_crs  = CRS.from_wkt(ds_nc.variables["spatial_ref"].crs_wkt)
dx = float(abs(x_nc[1] - x_nc[0]))
dy = float(abs(y_nc[0] - y_nc[1]))
nrows, ncols = len(y_nc), len(x_nc)
rain_tf = rasterio.transform.from_bounds(
    x_nc[0]  - dx/2, y_nc[-1] - dy/2,
    x_nc[-1] + dx/2, y_nc[0]  + dy/2,
    ncols, nrows,
)

# ── Reproject WGS84 → precipitation CRS ──────────────────────────────────────
tfm = Transformer.from_crs("EPSG:4326", precip_crs, always_xy=True)
rain_times = pd.Timestamp("2021-07-13 00:00") + pd.to_timedelta(time_h, unit="h")

def extract_rain_for_geom(geom_wgs84):
    """Extract mean rainfall over a WGS84 polygon via all_touched rasterization."""
    geom_proj = sh_transform(tfm.transform, geom_wgs84)
    mask = rasterio.features.rasterize(
        [(geom_proj, 1)],
        out_shape=(nrows, ncols),
        transform=rain_tf,
        fill=0,
        dtype=np.uint8,
        all_touched=True,
    ).astype(bool)
    n_px = mask.sum()
    if n_px == 0:
        cx, cy = geom_proj.centroid.x, geom_proj.centroid.y
        ix = int(np.clip(np.searchsorted(x_nc, cx), 0, ncols - 1))
        iy = int(np.clip(np.searchsorted(y_nc[::-1], cy), 0, nrows - 1))
        mask[iy, ix] = True
        n_px = 1
    vals = rain_mm[:, mask]   # shape (n_times, n_pixels)
    return pd.Series(np.nanmean(vals, axis=1), index=rain_times), n_px

for name, m in ungauged.items():
    m["rain_series_h"], n_px = extract_rain_for_geom(m["geom_wgs84"])
    rs = m["rain_series_h"]
    print(f"{name:<14}: {n_px:>3} pixels sampled | "
          f"peak = {rs.max():.1f} mm/h | event total = {rs.sum():.0f} mm")

print(f"\n  (if peak > ~100 mm/h, set rain_scale = 0.001 above)")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Method 1: Rainfall-weighted residual distribution + Kirpich injection lag
# ─────────────────────────────────────────────────────────────────────────────

sim_index  = df_obs.index   # 15-min DatetimeIndex, 2021-07-12 – 2021-07-17
total_area = sum(m["area_km2"] for m in ungauged.values())

# ── 1a. Hourly rainfall index covering the full simulation window ─────────────
hr_index = pd.date_range(t_start, t_end, freq="1h")

# ── 1b. Rainfall × area weights at HOURLY resolution ─────────────────────────
# Weights are computed at the native 1-hour resolution to avoid spurious
# sub-hourly oscillations: when fractions are computed on linearly-interpolated
# 15-min rain, small absolute differences create large relative swings that,
# multiplied by a large residual, produce spiky discharge.
weights_h = pd.DataFrame({
    name: m["rain_series_h"].reindex(hr_index).fillna(0.0) * m["area_km2"]
    for name, m in ungauged.items()
})
W_total_h      = weights_h.sum(axis=1)
# Fall back to area-only when total rain < 0.5 mm/h (noise floor)
W_total_h_safe = W_total_h.where(W_total_h > 0.5)

# ── 1c. Kirpich injection lag per tributary ────────────────────────────────────
# t_c [h] = 0.0663 × (L_km / sqrt(S))^0.77
# L_km and S measured along the AHR CHANNEL from each confluence to Altenahr
kirpich_params = {
    "New1":         {"L_km": 20.757847, "S": 0.00427},
    "Niederadenau": {"L_km": 14.231993, "S": 0.0045},
    "New2":         {"L_km":  12.42056, "S": 0.00482},
}

A_ref_km2 = 897.0   # total Ahr catchment area at Altenahr [km²]

print(f"{'Tributary':<14}  {'Kirpich [h]':>11}  {'GIUH-scaled [h]':>15}  {'Used (15-min steps)':>20}")
print("-" * 66)
for name, m in ungauged.items():
    p      = kirpich_params[name]
    lag_K  = 0.0663 * (p["L_km"] / p["S"]**0.5) ** 0.77
    lag_G  = lag_hours * (m["area_km2"] / A_ref_km2) ** 0.5
    m["lag_K_h"] = lag_K
    m["lag_G_h"] = lag_G
    m["lag_steps"] = max(1, round(lag_K * 4))
    print(f"{name:<14}  {lag_K:>10.2f}h  {lag_G:>14.2f}h  {m['lag_steps']:>15} steps")

# ── 1d. Distribute residual at hourly resolution, then resample to 15-min ─────
q_res_1h   = q_residual_smooth.reindex(hr_index, method="nearest")
df_method1 = df_obs[[1, 2, 4, 7]].copy()

for name, m in ungauged.items():
    wt_h = (weights_h[name] / W_total_h_safe).fillna(m["area_km2"] / total_area)
    # 3-hour centred smooth on the weight fraction to remove residual noise
    wt_h = wt_h.rolling(3, center=True, min_periods=1).mean()
    q_h  = (q_res_1h * wt_h).clip(lower=0)

    # Resample weighted discharge to 15-min, then apply Kirpich lag.
    # ffill fills the NaN tail left by shift(-n) with the last valid recession value.
    q_15 = (q_h.reindex(q_h.index.union(sim_index))
               .interpolate("time")
               .reindex(sim_index)
               .fillna(0.0))
    df_method1[m["index"]] = q_15.shift(-m["lag_steps"]).ffill().fillna(0.0).values

print(f"\nMethod 2 — peak discharge per ungauged tributary:")
for name, m in ungauged.items():
    print(f"  {name:<14}  {df_method1[m['index']].max():.2f} m³/s")


In [ ]:

# ── Method 1: plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)

for ax, (name, m) in zip(axes[:3], ungauged.items()):
    idx = m["index"]
    df_obs[idx].plot(ax=ax, label="M0 — area-only", color="gray", ls="--", alpha=0.8)
    df_method1[idx].plot(ax=ax, label="M1 — rain-weighted (Kirpich lag)", color="steelblue", lw=1.5)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("Q [m³/s]")
    ax.legend(fontsize=8)
    ax.grid(True)

ax = axes[3]
q_ahr.plot(ax=ax, label="Altenahr (observed)", color="k", lw=2.5)
df_obs.sum(axis=1).plot(ax=ax, label="Sum — M0 (area-only)", color="gray", ls="--", alpha=0.8)
df_method1.sum(axis=1).plot(ax=ax, label="Sum — M1 (rain-weighted)", color="steelblue", lw=1.5)
ax.set_ylabel("Q [m³/s]")
ax.set_xlabel("Time")
ax.set_title("Total inflow (gauged + ungauged) vs. Altenahr gauge")
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.suptitle("Method 1 — Rainfall-weighted residual distribution", y=1.01, fontsize=12)
plt.show()


---
## Method 2 — Event-based SCS-CN + Triangular Unit Hydrograph

Physically-based estimate of tributary discharge, **independent of the Altenahr residual**.

1. **Excess rainfall** — SCS Curve Number:  $Q_e = \dfrac{(P - 0.2S)^2}{P + 0.8S}$, with $S = \dfrac{25400}{CN} - 254$
2. **Time to peak** — Giandotti formula:  $T_p = \dfrac{4\sqrt{A} + 1.5\,L}{0.8\sqrt{\Delta H}}$ [h]
3. **Triangular unit hydrograph** — convolved with incremental excess rainfall
4. **Routing lag** — travel time from source outlet → Altenahr in SFINCS, approximated by the cross-correlation lag `best_lag`

**Action required**: update `A_km2`, `L_km`, `dH_m` and `CN` in `method2_params` with DEM-derived values and land-use-based CN (AMC-II, HSG B for mixed forest/agriculture in Eifel).

In [ ]:
import rasterio
from rasterio.mask import mask as rio_mask
import geopandas as gpd
import numpy as np
from pathlib import Path

# ── DEM tiles (UTM32N, 1 m resolution) ────────────────────────────────────────
dtm_dir   = project_root / "data" / "dtm" / "reprojected_32632"
dtm_files = sorted(dtm_dir.glob("*.tif"))

# Reproject catchment polygons to UTM32N (same CRS as DEM)
gdf_utm = gdf_all.to_crs("EPSG:32632")

# Source point coordinates in UTM32N (from src_points geojson)
outlet_coords_utm = {
    "New1":         (355971.43, 5597633.05),
    "Niederadenau": (353115.10, 5588924.27),
    "New2":         (352983.28, 5592048.43),
}

def mean_elevation_in_polygon(geom_utm, dtm_files):
    """Mean DEM elevation inside a UTM32N polygon (clips each tile separately)."""
    all_vals = []
    for f in dtm_files:
        with rasterio.open(f) as src:
            try:
                out, _ = rio_mask(src, [geom_utm], crop=True, nodata=src.nodata)
                data = out[0]
                valid = data[(data != src.nodata) & np.isfinite(data) & (data > -9000)]
                all_vals.extend(valid.tolist())
            except Exception:
                pass  # polygon doesn't overlap this tile
    return np.mean(all_vals) if all_vals else np.nan

print(f"{'Tributary':<14}  {'Mean elev [m]':>13}")
print("-" * 60)
for name, m in ungauged.items():
    comid    = gdf_all[abs(gdf_all["unitarea"] - m["area_km2"]) < 0.01]["COMID"].iloc[0]
    geom_utm = gdf_utm[gdf_utm["COMID"] == comid].geometry.iloc[0]

    mean_elev   = mean_elevation_in_polygon(geom_utm, dtm_files)
  
    print(f"{name:<14}  {mean_elev:>13.1f}")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Method 2: SCS-CN + Triangular Unit Hydrograph (Giandotti time to peak)
# ─────────────────────────────────────────────────────────────────────────────

# ── Catchment parameters ───────────────────────────────────────────────────────
# A_km2 : subcatchment area [km²]
# L_km  : longest flow path WITHIN the subcatchment [km]  ← update from DEM/OSM
# dH_m  : mean basin elevation − outlet elevation [m]     ← update from DEM
# CN    : SCS Curve Number, AMC-II (mixed forest/agriculture, HSG B; Eifel: 68–76)
method2_params = {
    "New1":         {"A_km2": 60.771605, "L_km": 17.281433 , "dH_m": 192.26, "CN": 85},
    "Niederadenau": {"A_km2": 61.338819, "L_km": 14.049829  , "dH_m": 212.455, "CN": 85},
    "New2":         {"A_km2": 29.453133, "L_km": 13.761004, "dH_m": 171.55, "CN": 85},
}

# ── Helper functions ───────────────────────────────────────────────────────────
def scs_excess_incremental(rain_h_arr, CN):
    """Incremental excess rainfall [mm/h] from hourly rain, using cumulative SCS-CN."""
    S  = 25400.0 / CN - 254.0
    Ia = 0.2 * S
    P  = np.cumsum(rain_h_arr)
    Qe_cum = np.where(P > Ia, (P - Ia)**2 / (P - Ia + S), 0.0)
    return np.maximum(np.diff(Qe_cum, prepend=0.0), 0.0)

def giandotti_tp(A_km2, L_km, dH_m):
    """Time to peak [h] at the subcatchment outlet — Giandotti formula."""
    return (4.0 * np.sqrt(A_km2) + 1.5 * L_km) / (0.8 * np.sqrt(dH_m))

def triangular_uh(A_km2, Tp_h, dt_h=1.0):
    """SCS triangular UH ordinates [m³/s per mm of excess rainfall]."""
    Tb  = 2.67 * Tp_h
    qp  = 0.208 * A_km2 / Tp_h
    t   = np.arange(0.0, Tb + dt_h, dt_h)
    q   = np.where(t <= Tp_h, qp * t / Tp_h,
              np.where(t < Tb, qp * (Tb - t) / (Tb - Tp_h), 0.0))
    return q.clip(min=0.0)

# ── Compute discharge at each tributary outlet ────────────────────────────────
# No routing lag applied: the hydrograph is already at the outlet.
# SFINCS routes it through the Ahr channel to Altenahr internally.
full_hr_idx = pd.date_range(t_start, t_end, freq="1h")
df_method2  = df_obs[[1, 2, 4, 7]].copy()

print(f"{'Tributary':<14}  {'Tp [h]':>7}  {'S [mm]':>7}  {'Peak Q [m³/s]':>14}")
print("-" * 50)

for name, p in method2_params.items():
    rain_h = ungauged[name]["rain_series_h"].reindex(full_hr_idx).fillna(0.0).values

    dQe = scs_excess_incremental(rain_h, p["CN"])
    Tp  = giandotti_tp(p["A_km2"], p["L_km"], p["dH_m"])
    uh  = triangular_uh(p["A_km2"], Tp)

    Q_h = np.convolve(dQe, uh)[: len(dQe)]
    Q_series_h = pd.Series(Q_h.clip(min=0.0), index=full_hr_idx)

    # Resample to 15-min — inject at outlet time, SFINCS routes to Altenahr
    Q_15 = (Q_series_h
            .reindex(Q_series_h.index.union(sim_index))
            .interpolate("time")
            .reindex(sim_index)
            .fillna(0.0))
    df_method2[ungauged[name]["index"]] = Q_15.values

    S_mm = 25400.0 / p["CN"] - 254.0
    print(f"{name:<14}  {Tp:>6.2f}h  {S_mm:>6.1f}mm  {Q_15.max():>13.2f}")


In [ ]:

# ── All-methods comparison plot + export ──────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(13, 13), sharex=True)

for ax, (name, m) in zip(axes[:3], ungauged.items()):
    idx = m["index"]
    df_obs[idx].plot(ax=ax, label="M1 — area-only", color="gray", ls="--", alpha=0.8, lw=1.2)
    df_method1[idx].plot(ax=ax, label="M2 — rain-weighted (Kirpich lag)", color="steelblue", lw=1.5)
    df_method2[idx].plot(ax=ax, label="M3 — SCS-CN + UH (Giandotti)", color="tomato", lw=1.5)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("Q [m³/s]")
    ax.legend(fontsize=8)
    ax.grid(True)

ax = axes[3]
q_ahr.plot(ax=ax, label="Altenahr (observed)", color="k", lw=2.5)
df_obs.sum(axis=1).plot(ax=ax, label="Sum — M1 (area-only)", color="gray", ls="--", alpha=0.8)
df_method1.sum(axis=1).plot(ax=ax, label="Sum — M2 (rain-weighted)", color="steelblue", lw=1.5)
df_method2.sum(axis=1).plot(ax=ax, label="Sum — M3 (SCS-CN)", color="tomato", lw=1.5)
ax.set_ylabel("Q [m³/s]")
ax.set_xlabel("Time")
ax.set_title("Total inflow (gauged + ungauged) vs. Altenahr gauge")
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.suptitle("All methods: comparison", y=1.01, fontsize=12)
plt.show()

# ── Export (fillna ensures no NaN reaches the SFINCS dis file) ───────────────
df_method1.fillna(0.0).to_csv(project_root / "data" / "observations" / "src_points_ts_method1.csv")
df_method2.fillna(0.0).to_csv(project_root / "data" / "observations" / "src_points_ts_method2.csv")
print("Exported: src_points_ts_method1.csv  (rainfall-weighted residual)")
print("          src_points_ts_method2.csv  (SCS-CN + triangular UH)")
